# MEN 4338 · Week 1 — Regionalised variables and exploratory data analysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W1_eda_bassam_gold.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

Geostatistics answers one question: **what is the grade where we have not drilled?** Before estimating
anything, an engineer looks hard at the data: how many samples, how they are spread in space, how the
grades are distributed, and whether a few very rich samples dominate the picture.

This semester we follow one deposit, the **Bassam gold prospect** (synthetic but realistic): a 10 m mining
bench, 1200 m × 1000 m, drilled with reverse-circulation (RC) holes on a 50 m grid, plus an infill
campaign at 25 m on the richest zone. Because the data are simulated, we also know the true grades
everywhere: later we will measure how good each estimation method really is.

**What you will practise:** loading data with pandas · location map · histogram and log-histogram ·
mean, median, variance, coefficient of variation · quantiles · the lognormal model · top-cutting (capping).

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('tests/w1/q1.py', 'tests/q1.py'), ('tests/w1/q2.py', 'tests/q2.py'), ('tests/w1/q3.py', 'tests/q3.py'), ('tests/w1/q4.py', 'tests/q4.py'), ('tests/w1/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. A regionalised variable

A grade $z(\mathbf{x})$ depends on the location $\mathbf{x} = (x, y)$: it is a **regionalised variable**.
Two samples close together tend to be similar, samples far apart are not. Classical statistics ignores
location; geostatistics uses it. First, the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("bassam_samples.csv")
print(df.shape)
df.head()

Each row is one RC hole: its 10 m bench **composite** grade `au_gpt` (g/t gold), the coordinates of the hole
in the mine grid (`x` east, `y` north, metres) and the `campaign` it belongs to (`grid` or `infill`).

In [ ]:
df.describe()

In [ ]:
df["campaign"].value_counts()

## 2. Where are the data?

Always start with a **location map**. Clusters of samples are a warning: if the infill holes were drilled
because the grades were good, they over-represent the rich zone.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for camp, marker in [("grid", "o"), ("infill", "^")]:
    d = df[df["campaign"] == camp]
    ax.scatter(d["x"], d["y"], s=14, marker=marker, label=camp)
ax.set_aspect("equal")
ax.set_xlabel("East (m)"); ax.set_ylabel("North (m)")
ax.legend(); ax.set_title("Bassam RC holes")
plt.show()

## 3. The distribution of grades

Gold grades are **positively skewed**: many low values, a few very high ones. The histogram of the
logarithm is often close to a bell curve: the grades are then said to be **lognormal**.

In [ ]:
au = df["au_gpt"].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(au, bins=40); axes[0].set_xlabel("Au (g/t)"); axes[0].set_title("Grades")
axes[1].hist(np.log(au), bins=40); axes[1].set_xlabel("ln Au"); axes[1].set_title("Log of grades")
plt.tight_layout(); plt.show()
print("skewness:", round(df["au_gpt"].skew(), 2))

Summary statistics you will use all semester:

| statistic | formula | meaning |
|---|---|---|
| mean | $\bar z = \frac{1}{n}\sum z_i$ | central value, sensitive to high grades |
| median | 50th percentile | central value, robust |
| variance | $s^2 = \frac{1}{n-1}\sum (z_i - \bar z)^2$ | spread (g/t)² |
| coefficient of variation | $CV = s / \bar z$ | relative spread; CV > 1 means very erratic grades |

A **probability plot** shows the same thing: lognormal data fall on a straight line when the grade axis is logarithmic.

In [ ]:
from scipy import stats
fig, ax = plt.subplots(figsize=(5, 4))
stats.probplot(np.log(au), dist="norm", plot=ax)
ax.set_title("Normal probability plot of ln Au")
plt.show()

---
## Exercises

Use `df` and `au` from above. Run `grader.check("q…")` under each answer.

### Q1. First numbers

Store the number of samples in `n_samples`, the mean grade in `mean_au` and the median grade in `median_au`.
Which one is larger, and why?

In [ ]:
n_samples = ...
mean_au = ...
median_au = ...
n_samples, mean_au, median_au

In [ ]:
grader.check("q1")

### Q2. Spread

Compute the sample variance `var_au`, the standard deviation `std_au` and the coefficient of variation
`cv_au` of the grades.

In [ ]:
var_au = ...
std_au = ...
cv_au = ...
var_au, std_au, cv_au

In [ ]:
grader.check("q2")

### Q3. Quantiles and the lognormal model

* `quantiles`: the 10th, 25th, 50th, 75th and 90th percentiles of the grades (one call to `np.percentile`).
* `iqr_au`: the inter-quartile range (75th − 25th percentile).
* `log_mean`, `log_var`: mean and sample variance of $\ln z$.
* `lognormal_mean`: the mean predicted by the lognormal model, $e^{\mu + \sigma^2/2}$ with
  $\mu$ = `log_mean` and $\sigma^2$ = `log_var`. Compare it with `mean_au`.

In [ ]:
quantiles = ...
iqr_au = ...
log_mean = ...
log_var = ...
lognormal_mean = ...
quantiles, iqr_au, lognormal_mean, mean_au

In [ ]:
grader.check("q3")

### Q4. Capping the outliers

A few extreme assays can inflate the estimated metal. A common practice is to **cap** (top-cut) grades at a
high percentile. Cap at the 99th percentile:

* `cap_value`: the 99th percentile of the grades,
* `au_capped`: the grades with every value above `cap_value` replaced by `cap_value`,
* `n_capped`: how many samples were capped,
* `metal_removed`: the fraction of the total metal removed, $1 - \sum z^{cap} / \sum z$.

In [ ]:
cap_value = ...
au_capped = ...
n_capped = ...
metal_removed = ...
cap_value, n_capped, metal_removed

In [ ]:
grader.check("q4")

### Q5. Where is the gold?

Store in `prop_above_1` the share of samples above 1 g/t. Then make a map of the samples coloured by grade
in `fig_map, ax_map`: `ax_map.scatter(x, y, c=au, cmap="viridis")`, a colour bar, equal aspect, and both axes
labelled with their unit. Where is the rich zone? Does it match the infill holes?

In [ ]:
prop_above_1 = ...

fig_map, ax_map = plt.subplots(figsize=(7, 6))
...
plt.show()
prop_above_1

In [ ]:
grader.check("q5")

### Going further (not checked)

* Compare the mean grade of the `grid` holes with that of the `infill` holes. Which one better represents the whole bench?
* Plot the histogram with `bins=np.logspace(-1, 1, 40)` and `ax.set_xscale("log")`.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()